# NixLMs Neural Core 0.4 — Semantic Learning — Google Colab
Encoder treinável por pares contrastivos, memória vetorial híbrida e rede LIF experimental. O núcleo roda em CPU sem dependências Python extras.

In [ ]:
!git clone -b feature/semantic-learning-0.4 https://github.com/SANTZSPAXX/NixLMs.git
%cd NixLMs
from nixlms_core import NixLMsNeuralCore
ia = NixLMsNeuralCore()
if not ia.learning.dataset.load():
    print('Dataset de exemplo importado:', ia.learning.dataset.import_jsonl('datasets/starter_ptbr.jsonl'))
print(ia.status())

## LLM local opcional no Colab
Se quiser experimentar geração com um modelo local dentro da sessão Colab, execute a célula abaixo. Ela instala Ollama e baixa Qwen 2.5 0.5B; isso usa internet, armazenamento e RAM/GPU da sessão. Não é necessário para usar os outros módulos.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve > /tmp/ollama.log 2>&1 &
!sleep 5
!ollama pull qwen2.5:0.5b
print('Ollama disponível:', ia.language.available())

## Conversa
Se Ollama estiver ativo e o modelo baixado, a NixLMs usa o LLM local. Caso contrário, faz fallback para memória e recuperação.

In [ ]:
while True:
    mensagem = input('Você: ').strip()
    cmd = mensagem.lower()
    if cmd in {'/sair','/exit','/quit'}:
        print('Até logo!'); break
    elif cmd == '/ajuda':
        print('/ensinar pergunta => resposta | /ensinar-par texto A || texto B || 1|0 | /status | /mapa | /treinar [epocas] | /avaliar | /sair')
    elif cmd == '/status':
        print(ia.status())
    elif cmd == '/mapa':
        print({'nodes':len(ia.mental_map.nodes),'edges':len(ia.mental_map.edges),'related':ia.mental_map.associate(ia.last_question,10)})
    elif cmd == '/avaliar':
        print(ia.learning.evaluate())
    elif cmd.startswith('/treinar'):
        p=cmd.split(); print(ia.learning.train(int(p[1]) if len(p)>1 else 1))
    elif cmd.startswith('/ensinar-par '):
        p=[x.strip() for x in mensagem[len('/ensinar-par '):].split('||')]
        if len(p)!=3: print('Use /ensinar-par texto A || texto B || 1 (ou 0)')
        else:
            ia.learning.dataset.add_pair(p[0],p[1],int(p[2]),'train')
            ia.encoder.fit_vocabulary(p[:2]); ia._save_encoder()
            print('Par contrastivo salvo; execute /treinar.')
    elif cmd.startswith('/ensinar '):
        payload=mensagem[len('/ensinar '):]
        if '=>' not in payload: print('Use /ensinar pergunta => resposta')
        else:
            q,a=payload.split('=>',1); ia.learn(q,a,add_to_dataset=True)
            print('Conhecimento salvo.')
    else:
        print('NixLMs:',ia.respond(mensagem,use_language=True))

## Testes automatizados
A célula executa os testes da versão sem depender do Ollama.

In [ ]:
!python -m unittest discover -s tests -v